# Introduction

In this notebook, we are going to gradually build "context" for a story about John Snow's cholera map. We will combine text, pictures, and visualizations to show that the 'myth' of John Snow is a bit more nuanced than most people are led to believe.

## John Snow's Cholera Map

<p><img style="float: left;margin:5px 20px 5px 1px" src="https://i.guim.co.uk/img/static/sys-images/Guardian/Pix/pictures/2013/3/14/1363295337709/johnsnowillustration.png?width=700&quality=85&auto=format&fit=max&s=061b5c30d90beaa099a0863b2bc505de"></p>

<p>Dr. John Snow (1813-1858) is regarded as a crucial figure in the history of epidemiology and public health. Among other things, he is credited with using maps to demonstrate that the clusters of deaths in London's Soho during an 1854 cholera outbreak were caused by contaminated water. This marked a major shift in thinking away from the disease being transmitted through dirty air - the 'miasma theory' - which was the prevailing view at the time. Snow's innovative methods retain influence in fields ranging from geography to data science, and his cholera map is now a pillar of data visualization curriculum.</p>

<p><img style="float: left;margin:5px 20px 5px 1px" src="https://s3.amazonaws.com/assets.datacamp.com/production/project_132/img/johnsnow_original.jpg"> </p>



First, we will set up our notebook environment.

In [1]:
# Import the necessary libraries
# You will likely need to install a number of these modules/libraries
# (e.g. pip install folium or conda install geopandas)
import pandas as pd
import folium
from shapely.geometry import Point, Polygon

In [2]:
# This is unnecessary if you plan to run the notebook from JupyterLab (locally, or from the OTU hub at https://hub.science.ontariotechu.ca/)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Next, we will load some data and try to make sense of it.

In [4]:
# Read in the data (make sure you download each file and place them in a "data" directory where your .ipynb is located)
deaths = pd.read_csv('/content/drive/Shareddrives/MBAI 5400G - Visualization and Storytelling (Winter 2026)/In-Class Activities/Python Visualization/data/deaths.csv')
pumps = pd.read_csv('/content/drive/Shareddrives/MBAI 5400G - Visualization and Storytelling (Winter 2026)/In-Class Activities/Python Visualization/data/pumps.csv')

In [5]:
# Take a quick look at the deaths data
deaths.head()

,death_count,x_latitude,y_longitude
0,1,51.513418,-0.137930
1,1,51.513418,-0.137930
2,1,51.513418,-0.137930
3,1,51.513361,-0.137883
4,1,51.513361,-0.137883


In [6]:
# Create 'locations' variables by subsetting only Latitude and Longitude from the datasets
locations_deaths = deaths[['x_latitude', 'y_longitude']]
locations_pumps = pumps[['x_coordinate', 'y_coordinate']]

# Transform the dataframes to list of lists
deaths_list = locations_deaths[['x_latitude', 'y_longitude']].values.tolist()
pumps_list = locations_pumps[['x_coordinate', 'y_coordinate']].values.tolist()

Next, we will use Folium, a Python interface for leaflet maps, to plot the data on a realistic map of London.

https://python-visualization.github.io/folium/latest/


In [7]:
# (change "tile" to try a different background and folium marker to try different methods of representation)
map = folium.Map(location=[51.5132119,-0.13666], tiles='Cartodb Positron', zoom_start=17)
for point in range(0, len(locations_deaths)):
    folium.CircleMarker(deaths_list[point], radius=8, color='black', fill=True, fill_color='black', opacity = 0.4).add_to(map)
map1 = map
for point in range(0, len(locations_pumps)):
    folium.Marker(pumps_list[point], popup=pumps['pump_name'][point]).add_to(map1)

# Display the map
map1

How does the data look when clustered?

In [8]:
from folium.plugins import FastMarkerCluster
FastMarkerCluster(data=list(zip(deaths['x_latitude'].values, deaths['y_longitude'].values))).add_to(map)
folium.LayerControl().add_to(map)
map

Maybe we should style the clusters...

In [9]:
for index, row in deaths.iterrows():

    folium.CircleMarker(location=(row["x_latitude"],
                                  row["y_longitude"]),
                        radius= row['death_count']*10,
                        color='#FA8072',
                        fillOpacity=0.9,
                        fill=True).add_to(map)
map

Let's make a heatmap with a watercolor background!

In [10]:
base_map = folium.Map(location=[51.5132119,-0.13666], tiles='https://watercolormaps.collection.cooperhewitt.org/tile/watercolor/{z}/{x}/{y}.jpg',attr='Map tiles by Stamen Design, under CC BY 3.0. Data by OpenStreetMap, under CC BY SA.' , zoom_start=16)

for i,row in pumps.iterrows():
    folium.Marker([row['x_coordinate'],row['y_coordinate']], popup=row['pump_name']).add_to(base_map)
base_map
heat_data = [[row['x_latitude'],row['y_longitude']] for index, row in locations_deaths.iterrows()]

In [11]:
from folium import plugins
from folium.plugins import HeatMap
HeatMap(heat_data).add_to(base_map)
base_map